# Day 51 Practice: How LLM Apps Work

Work through the exercises in order and run each cell with **Shift+Enter**.
Write your code where it says `# your code here`. No API key is needed today.

✅ Exercises 1–4: everyone · Exercise 5: most of you · Exercise 6: challenge

In [ ]:
import csv
import random
from collections import Counter
from pathlib import Path

## Exercise 1: Predict the next word

The cell below "trains" a toy next-word model on seven sentences (the same idea as the demo).
Run it, then **write your predictions** before running the next cell.

In [ ]:
corpus = [
    "i study python every day",
    "i study python at night",
    "i study maths every week",
    "i love python",
    "python is fun",
    "python is fun",
    "python is hard",
]

next_words = {}
for sentence in corpus:
    words = sentence.split()
    for i in range(len(words) - 1):
        if words[i] not in next_words:
            next_words[words[i]] = []
        next_words[words[i]].append(words[i + 1])


def predict_next(word):
    """Return the word most often seen after `word`."""
    return Counter(next_words[word]).most_common(1)[0][0]


def generate(start, max_words=8):
    """Build a sentence by repeatedly predicting the next word."""
    words = [start]
    while words[-1] in next_words and len(words) < max_words:
        words.append(predict_next(words[-1]))
    return " ".join(words)

My predictions:
- `predict_next("i")` → ...
- `predict_next("python")` → ...
- `generate("i")` → ...
- With `random.choice(next_words["study"])` (a "temperature above 0" choice), how often do you expect `"maths"`? ...

In [ ]:
# Check your predictions here, e.g. print(predict_next("i"))
# your code here

## Exercise 2: Messages and tokens

You are building a "study buddy" app. Create:

- `system`: a string telling the model it is a patient Python tutor who answers in at most 3 sentences.
- `messages`: a list of **three** message dicts for this conversation:
  1. user: `What is a list?`
  2. assistant: `A list stores several values in order, like [1, 2, 3].`
  3. user: `How do I add an item?`

Then print each message as `role: content`, and print the estimated number of tokens of the system text plus all
message contents (use `estimate_tokens`).

Expected output (your token number depends on your system text):
```
user: What is a list?
assistant: A list stores several values in order, like [1, 2, 3].
user: How do I add an item?
Estimated tokens: 39
```

In [ ]:
def estimate_tokens(text):
    """Rough token estimate for English text: about 4 characters per token."""
    return round(len(text) / 4)


# your code here

In one sentence: why does the app send the first two messages again, when the model already answered them?

*Your answer:*

## Exercise 3: A prompt template

Write a function `build_summary_prompt(product, review, max_words)` that **returns** this text, with the three
values filled in:

```
You summarise customer reviews for an online shop.
Summarise this review of a {product} in at most {max_words} words.

Review: {review}
```

The check below must print `True`.

In [ ]:
# your code here


expected = """You summarise customer reviews for an online shop.
Summarise this review of a desk lamp in at most 10 words.

Review: Flickers constantly. Gave me a headache."""
# print(build_summary_prompt("desk lamp", "Flickers constantly. Gave me a headache.", 10) == expected)

## Exercise 4: Fix the pipeline

This pipeline has **three** bugs. None of them is a typo in a name. Copy it into the cell below, fix it and run
it. When it is fixed, the last line prints `True`.

```python
def build_prompt(review):
    return "Is this review positive or negative? Answer 'Sentiment: <label>'.\nReview: {review}"


def fake_llm(prompt):
    if "love" in prompt.lower():
        print("Sentiment: Positive")
    else:
        print("Sentiment: Negative")


def parse_sentiment(reply):
    return reply.split(":")[1]


label = parse_sentiment(fake_llm(build_prompt("I love this backpack")))
print(label == "positive")
```

In [ ]:
# fixed version

## Exercise 5: The pipeline on real reviews

The cell below contains the demo's three pipeline functions. Use them to analyse every **desk lamp** review in
`reviews.csv`:

1. For each desk lamp review print the ID, the stars and the label from the pipeline.
2. Count how many times each label appears (a dict).
3. Count how often the label agrees with the stars (4–5 → positive, 1–2 → negative, 3 → neutral) and print it.

Expected output:
```
R16 5 positive
R17 1 negative
R18 4 neutral
...
Label counts: {'positive': 3, 'negative': 1, 'neutral': 4}
Agrees with the stars on 3 of 8 desk lamp reviews
```

In [ ]:
POSITIVE = ["great", "love", "comfortable", "perfect", "easy", "recommended", "best", "good"]
NEGATIVE = ["broke", "leaks", "cracked", "terrible", "flickers", "disappointed", "poor", "stopped"]
ALLOWED = ["positive", "negative", "neutral"]


def build_prompt(review):
    """Fill the sentiment prompt template with one review."""
    return f"""Classify the sentiment of this product review.
Answer in the form "Sentiment: Positive", "Sentiment: Negative" or "Sentiment: Neutral".

Review: {review}"""


def fake_llm(prompt):
    """A pretend LLM: counts positive and negative words in the prompt."""
    cleaned = prompt.lower().replace(".", " ").replace(",", " ").replace("!", " ")
    score = 0
    for word in cleaned.split():
        if word in POSITIVE:
            score += 1
        elif word in NEGATIVE:
            score -= 1
    if score > 0:
        return "Sentiment: Positive"
    if score < 0:
        return "Sentiment: Negative"
    return "Sentiment: Neutral"


def parse_sentiment(reply):
    """Turn 'Sentiment: Positive' into 'positive'. Unexpected replies become 'unknown'."""
    if ":" not in reply:
        return "unknown"
    label = reply.split(":")[1].strip().lower()
    if label not in ALLOWED:
        return "unknown"
    return label


reviews_path = Path("../../../datasets/reviews.csv")

In [ ]:
# your code here

## Exercise 6 (challenge): Teach the fake model "not"

Reviews such as *"Not happy."* or *"the plug did not fit"* fool a keyword counter.

1. Write `fake_llm_v2(prompt)`: like `fake_llm`, but uses the longer word lists below, and a positive word that
   comes **right after `"not"`** counts as negative (`not happy` → −1).
2. Measure both models on **all 40 reviews**: how many labels agree with the stars? Use the lists below for
   *both* models so the comparison is fair (hint: give your functions the word list as a parameter, or write a
   `fake_llm_v1` that uses the long lists without the "not" rule).

Expected output:
```
without the 'not' rule: 27 of 40
with the 'not' rule:    29 of 40
```

In [ ]:
POSITIVE_LONG = POSITIVE + ["happy", "fit", "nice", "solid", "clear"]
NEGATIVE_LONG = NEGATIVE + ["wrong", "late", "ripped", "peeling"]

# your code here

Write one sentence: why is "measure before and after" better than trying three reviews by hand?

*Your answer:*